  #                                                                       # *****DATA PRE PROCESSING*****

# ***Demography***

In [ ]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.simplefilter("ignore", UserWarning)

In [ ]:
# from pathlib import Path
# data_folder = Path(
#     r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\Python_Hackathon_Sep_2026\Python_Hackathon_Sep_2026\cardiac_failure")

### **1. Read the CSV file and inspect the data. This confirms the file loaded correctly and shows missing values before you change anything**

In [ ]:
# Read the demography CSV File and inspect the data 

dfDEMO = pd.read_csv("data_files/demography.csv")
df_original = dfDEMO.copy()

print("Rows and columns:", dfDEMO.shape)
display(dfDEMO.head())
dfDEMO.info()

print("\nMissing values:")
display(dfDEMO.isna().sum())

print(
    "Duplicate patient IDs:",
    dfDEMO["inpatient_number"].duplicated().sum()
)



### **2. Removed the empty record. Patient ID 5 has no gender, weight, height,occupation, or age category. Its BMI value alone is not enuogh to use the record for demographic analysis. So this removes one row**


In [ ]:
dfDEMO = dfDEMO.dropna(
    subset=["gender", "weight", "height", "occupation", "agecat"],
    how="all"
).copy()

### **3. Renamed column names for best readability and consistency across datasets for less confusion during analysis**

In [ ]:
dfDEMO = dfDEMO.rename(columns={
    "inpatient_number": "patient_id",
    "agecat" : "age_category"
})

### **4. Removed extra spaces from the text columns so values such as "Male" and " Male " are treated as the same category. I also make the occupation names consistent and easier to read. This prevents one category from appearing under different labels in charts and counts.**

In [ ]:
print(dfDEMO.columns.tolist())

text_columns = ["gender", "occupation", "age_category"]

for column in text_columns:
    dfDEMO[column] = dfDEMO[column].astype("string").str.strip()

dfDEMO["occupation"] = dfDEMO["occupation"].replace({
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "worker": "Worker"
})

dfDEMO["occupation"] = dfDEMO["occupation"].fillna("Unknown")



### **5. Impossible measurements, Weight values of zero or less and height values below 1.0 are flagged for review. The code marks those rows in measurement and changes the flagged values to missing so they do not affect BMI calculations. It then counts how many records were flagged.***

In [ ]:
dfDEMO["measurement"] = pd.Series(False, index=dfDEMO.index)

invalid_weight = dfDEMO["weight"] <= 0
invalid_height = dfDEMO["height"] < 1.0

dfDEMO.loc[invalid_weight | invalid_height, "measurement"] = True

dfDEMO.loc[invalid_weight, "weight"] = np.nan
dfDEMO.loc[invalid_height, "height"] = np.nan

print("Records with measurement issues:", dfDEMO["measurement"].sum())

### **6. Recalculated BMI from Valid measurements. The orginial BMI values match the recorded weights and heights, including the incorrect heights and zero weights, Recalculating after flagging those measurements makes BMI missing for the seven affected records**

In [ ]:
dfDEMO["bmi_original"] = dfDEMO["bmi"].round(2)

dfDEMO["bmi"] = dfDEMO["weight"] / (dfDEMO["height"] ** 2)
dfDEMO["bmi"] = dfDEMO["bmi"].round(2)


### **7. Removed Unrealistic BMI values and validated age category consistency**

In [ ]:
dfDEMO["bmi"] = pd.to_numeric(dfDEMO["bmi"], errors="coerce")
dfDEMO.loc[(dfDEMO["bmi"] < 15) | (dfDEMO["bmi"] > 60), "bmi"] = np.nan
dfDEMO["age_category"] = dfDEMO["age_category"].astype(str).str.strip()
dfDEMO["age_category"] = dfDEMO["age_category"].str.title()

### **8. Reviewed the cleaned data to make sure it is ready for analysis. I check the number of rows and unique patients, see which values are still missing, review the weight, height, and BMI ranges, and count each category. Finally, I display the records flagged for measurement issues so I can inspect them before using them in calculations.**

In [ ]:
print("Rows:", len(dfDEMO))
print("Unique patient IDs:", dfDEMO["patient_id"].nunique())
print("\nMissing values:")
print(dfDEMO.isna().sum())

print("\nNumeric summary:")
print(dfDEMO[["weight", "height", "bmi"]].describe())

print("\nCategory counts:")
for column in ["gender", "occupation", "age_category"]:
    print(f"\n{column}")
    print(dfDEMO[column].value_counts(dropna=False))

print("\nRecords needing measurement review:")
print(
    dfDEMO.loc[
        dfDEMO["measurement"],
        ["patient_id", "weight", "height", "bmi_original"]
    ]
)

### **9. Save analysis and reviewing the files**

In [ ]:
dfDEMO.to_csv("cleaned_files/demography_clean.csv", index=False)

#                                                          ***Patient History***

### **1. Load the CSV File and check the size. column types, missing values, and patient IDs before changing anything**

In [ ]:
dfPH = pd.read_csv("data_files/patienthistory.csv")
df_original = dfPH.copy()

print("Rows and columns:", dfPH.shape)
display(dfPH.head())
dfPH.info()

print("\nMissing values:")
display(dfPH.isna().sum())

print(
    "Duplicate patient IDs:",
    dfPH["inpatient_number"].duplicated().sum()
)


### ***2. Renamed the column name inpatient_number to patient_id***

In [ ]:
dfPH = dfPH.rename(columns={
    "inpatient_number": "patient_id",
    "type_ii_respiratory_failure": "respiratory_failure_type2",
    "moderate_to_severe_chronic_kidney_disease": "mod_severe_ckd",
    "chronic_obstructive_pulmonary_disease": "copd"
})

### ***3. Condition should check only 0,1, or missing values reviewing unexpected values before converting Types***

In [ ]:
condition_columns = [
    "cerebrovascular_disease",
    "dementia",
    "copd",
    "connective_tissue_disease",
    "peptic_ulcer_disease",
    "diabetes",
    "mod_severe_ckd",
    "hemiplegia",
    "leukemia",
    "malignant_lymphoma",
    "solid_tumor",
    "liver_disease",
    "aids",
    "acute_renal_failure",
]

for column in condition_columns:
    print(f"\n{column}")
    print(dfPH[column].value_counts(dropna=False))

### ***4. Consistent text labels prevent the same category from appearing under different names in charts***

In [ ]:
dfPH["respiratory_failure_type2"] = (
    dfPH["respiratory_failure_type2"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(dfPH["respiratory_failure_type2"].value_counts(dropna=False))

### ***5. cci_score can be used to compare groups of patients, so check its range and missing values. Do not invent scores for patients whose values are missing***

In [ ]:
dfPH["cci_score"] = pd.to_numeric(
    dfPH["cci_score"],
    errors="coerce"
).astype("Int64")

print(dfPH["cci_score"].value_counts(dropna=False).sort_index())
print("Missing CCI scores:", dfPH["cci_score"].isna().sum())
dfPH["cci_score"].describe()

### ***6. Checking the cleaned file and saving the file. To confirm that cleaning did not accidentally remove patients or create duplicate IDs before joining this file to the other datasets***

In [ ]:
print("Original rows:", len(df_original))
print("Cleaned rows:", len(dfPH))
print("Unique patient IDs:", dfPH["patient_id"].nunique())
print("Duplicate patient IDs:", dfPH["patient_id"].duplicated().sum())

print("\nRemaining missing values:")
display(dfPH.isna().sum())

assert len(dfPH) == 2008
assert dfPH["patient_id"].is_unique


dfPH.to_csv(
    "cleaned_files/patienthistory_clean.csv",
    index=False
)

print("Saved patienthistory_clean.csv")

# ***Patient Prescriptions***

### **1. Read the CSV file and inspect the data. This confirms the file loaded correctly and shows missing values before you change anything**

In [ ]:
import pandas as pd
import numpy as np

precriptions = pd.read_csv("data_files/patient_precriptions.csv")

# 1) Basic inspection
print(precriptions.shape)
print(precriptions.head())
print(precriptions.isna().sum())
print("Duplicate rows:", precriptions.duplicated().sum())
print("Duplicate patient-drug pairs:", precriptions.duplicated(subset=["inpatient_number", "drug_name"]).sum())


### **2. Rename the patient ID to be consistent across multiple data files**

In [ ]:
# 2) Clean patient ID
precriptions = precriptions.rename(columns={'inpatient_number': 'patient_id'})

### **3. Standardize the drug_name column values - removing unnecessary white spaces and lower the words to be in consistent with other values.**

In [ ]:
# 3) Clean drug names
precriptions["drug_name"] = precriptions["drug_name"].astype(str).str.strip()
precriptions["drug_name"] = precriptions["drug_name"].str.replace(r"\s+", " ", regex=True)
precriptions["drug_name"] = precriptions["drug_name"].str.lower()

### **4. Check for any missing drug entries or with blank names. These records can be removed as they does not add any value to the dataset when performing descriptive or predective analysis to the model.**

In [ ]:
# 4) Remove blank or missing drug entries
precriptions = precriptions[precriptions["drug_name"].notna() & precriptions["drug_name"].str.strip().ne("")]


### **5. Save the cleaned prescriptions data file**

In [ ]:
precriptions.to_csv("cleaned_files/prescriptions_clean.csv", index=False)
print("Saved prescriptions_clean.csv")

# ***Labs***

### **1. Load and inspect the file**


In [ ]:
import pandas as pd
import numpy as np


labs = pd.read_csv("data_files/labs.csv")

print(labs.shape)       # 2,008 rows, 107 columns
display(labs.head())
labs.info()


### **2. Check missing values and duplicates. This file has no duplicate rows or patient IDs. Many lab columns have missing values because a test was not recorded for every patient. Do not replace all those blanks with zero.**



In [ ]:
missing = labs.isna().sum().sort_values(ascending=False)
print(missing)

print("Duplicate rows:", labs.duplicated().sum())
print("Duplicate patient IDs:", labs["inpatient_number"].duplicated().sum())


### **3. Remove the completely empty column. The Cholinesterase column is completely empty and hence cannot be used for further analysis. So dropping the column from labs.**



In [ ]:
empty_columns = labs.columns[labs.isna().all()]
print(empty_columns.tolist())  # ['cholinesterase']

labs = labs.drop(columns=empty_columns)


### **4. Mark unusable vital signs as missing. Three rows have blood pressure recorded as zero; two have systolic pressure below diastolic pressure. Mark the blood pressure values and their calculated map_value as missing in those five rows.**



In [ ]:
invalid_bp = (
    (labs["systolic_blood_pressure"] == 0) |
    (labs["diastolic_blood_pressure"] == 0) |
    (labs["systolic_blood_pressure"] < labs["diastolic_blood_pressure"])
)

print("Rows with invalid blood pressure:", invalid_bp.sum())

labs.loc[
    invalid_bp,
    ["systolic_blood_pressure", "diastolic_blood_pressure", "map_value"]
] = np.nan

labs.loc[labs["pulse"] == 0, "pulse"] = np.nan
labs.loc[labs["respiration"] == 0, "respiration"] = np.nan


In [ ]:
keep_cols = [
    "inpatient_number",
    "body_temperature",
    "pulse",
    "respiration",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "fio2",
    "creatinine_enzymatic_method",
    "urea",
    "uric_acid",
    "glomerular_filtration_rate",
    "cystatin",
    "white_blood_cell",
    "monocyte_count",
    "red_blood_cell",
    "hematocrit",
    "lymphocyte_count",
    "hemoglobin",
    "platelet",
    "basophil_count",
    "eosinophil_count",
    "neutrophil_count",
    "activated_partial_thromboplastin_time",
    "fibrinogen",
    "high_sensitivity_troponin",
    "myoglobin",
    "calcium",
    "potassium",
    "chloride",
    "sodium",
    "serum_magnesium",
    "creatine_kinase",
    "lactate_dehydrogenase",
    "brain_natriuretic_peptide",
    "albumin",
    "globulin",
    "total_protein",
    "cholesterol",
    "triglyceride",
    "glucose_blood_gas",
    "lactate",
    "partial_oxygen_pressure",
    "ph",
    "potassium_ion",
    "chloride_ion",
    "sodium_ion",
    "free_calcium"
]



### **5. Drop these derived / redundant columns**

***1. Derived values should not dominate the dataset***

Columns like:

map_value,
mean_corpuscular_volume,
mean_hemoglobin_volume,
mean_hemoglobin_concentration,
standard_bicarbonate,
anion_gap. 
are typically calculated from more basic variables. If you already have the raw values, these can be recreated later in analysis or feature engineering.

***2. Ratios and percentages are often redundant***

Variables such as:

monocyte_ratio,
basophil_ratio,
eosinophil_ratio,
neutrophil_ratio,
white_globulin_ratio  
are often informative, but they are not always necessary when the absolute count variables are already present. A model can always derive ratios later if needed.

***3. Blood-gas “derived” acid-base variables are not primary features***

These variables are often used for clinical interpretation, but for a simplified modeling dataset they are less useful than the direct measures:

pH,
partial_oxygen_pressure,
partial_pressure_of_carbon_dioxide,
lactate,
glucose_blood_gas 

So keeping the direct values and dropping the computed acid-base summaries is a sensible choice.

***4. Some columns are duplicated in meaning***

Examples:

total_protein and globulin are both part of protein balance,
mean_platelet_volume and platelet are different conceptually, but the calculated index is often less valuable for a reduced dataset,
international_normalized_ratio and prothrombin_time_ratio are related to the same coagulation process.
These are good candidates for dropping when working with a smaller variable set.

***5. A smaller dataset is easier to model and interpret***

Your goal is not to keep every possible field, but to keep:

stable raw measurements,
clinically important markers,
variables that are interpretable without complex calculations,
This makes the dataset easier to clean, visualize, and model without creating noise from highly derived features


In [ ]:
drop_cols = [
    "map_value",
    "monocyte_ratio",
    "basophil_ratio",
    "eosinophil_ratio",
    "neutrophil_ratio",
    "mean_corpuscular_volume",
    "mean_hemoglobin_volume",
    "mean_hemoglobin_concentration",
    "mean_platelet_volume",
    "platelet_distribution_width",
    "platelet_hematocrit",
    "coefficient_of_variation_of_red_blood_cell_distribution_width",
    "standard_deviation_of_red_blood_cell_distribution_width",
    "international_normalized_ratio",
    "prothrombin_time_ratio",
    "prothrombin_activity",
    "white_globulin_ratio",
    "standard_residual_base",
    "measured_residual_base",
    "standard_bicarbonate",
    "measured_bicarbonate",
    "total_carbon_dioxide",
    "anion_gap",
    "oxygen_saturation",
    "oxyhemoglobin",
    "carboxyhemoglobin",
    "methemoglobin",
    "hematocrit_blood_gas",
    "total_hemoglobin",
    "hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase",
    "creatine_kinase_isoenzyme_to_creatine_kinase",
    "total_bile_acid",
    "total_protein",
    "globulin"
]

In [ ]:
labs = labs.drop(columns=drop_cols, errors="ignore")

### **6. Rename the columns for the reduced labs dataframe. Keep the names short, consistent, and clinically readable. This helps later modeling and makes the notebook easier to follow. For example:**

**Short and readable:** systolic_bp, diastolic_bp, gfr, wbc, po2\
**Consistent format:** all names are lowercase snake_case\
**Medical convention:** aptt, bnp, ck, ldh, hs_troponin\
**Clearer for analysis:** patient_id instead of inpatient_number




In [ ]:
rename_map = {
    "inpatient_number": "patient_id",
    "body_temperature": "body_temp",
    "pulse": "pulse",
    "respiration": "respiration",
    "systolic_blood_pressure": "systolic_bp",
    "diastolic_blood_pressure": "diastolic_bp",
    "fio2": "fio2",
    "creatinine_enzymatic_method": "creatinine",
    "urea": "urea",
    "uric_acid": "uric_acid",
    "glomerular_filtration_rate": "gfr",
    "cystatin": "cystatin",
    "white_blood_cell": "wbc",
    "monocyte_count": "monocyte_count",
    "red_blood_cell": "rbc",
    "hematocrit": "hematocrit",
    "lymphocyte_count": "lymphocyte_count",
    "hemoglobin": "hemoglobin",
    "platelet": "platelet_count",
    "basophil_count": "basophil_count",
    "eosinophil_count": "eosinophil_count",
    "neutrophil_count": "neutrophil_count",
    "activated_partial_thromboplastin_time": "aptt",
    "fibrinogen": "fibrinogen",
    "high_sensitivity_troponin": "hs_troponin",
    "myoglobin": "myoglobin",
    "calcium": "calcium",
    "potassium": "potassium",
    "chloride": "chloride",
    "sodium": "sodium",
    "serum_magnesium": "magnesium",
    "creatine_kinase": "ck",
    "lactate_dehydrogenase": "ldh",
    "brain_natriuretic_peptide": "bnp",
    "albumin": "albumin",
    "globulin": "globulin",
    "total_protein": "total_protein",
    "cholesterol": "cholesterol",
    "triglyceride": "triglyceride",
    "glucose_blood_gas": "glucose_blood_gas",
    "lactate": "lactate",
    "partial_oxygen_pressure": "po2",
    "ph": "ph",
    "potassium_ion": "potassium_ion",
    "chloride_ion": "chloride_ion",
    "sodium_ion": "sodium_ion",
    "free_calcium": "free_calcium",
    "partial_pressure_of_carbon_dioxide": "paco2"
}

labs = labs.rename(columns=rename_map)

In [ ]:
labs.describe()
labs.isna().sum()
display(labs.dtypes)

### **7. Flag impossible or clinically unrealistic values. This catches data-entry issues and impossible physiology values. It is often more useful than blindly dropping rows.**
There are likely other impossible values across the dataset.

Examples:

temperature < 30 or > 45\
pulse < 20 or > 200\
respiration < 4 or > 60\
sodium < 100 or > 180\
potassium < 1 or > 8\
creatinine < 0

In [ ]:
for col, low, high in [
    ("body_temp", 30, 45),
    ("pulse", 20, 200),
    ("respiration", 4, 60),
    ("sodium", 100, 180),
    ("potassium", 1, 8),
    ("creatinine", 0, 20)
]:
    if col in labs.columns:
        labs.loc[(labs[col] < low) | (labs[col] > high), col] = np.nan

### **8. Check and standardize column types. Some lab values may still be read as strings, especially if there are commas, units, or blanks.Numeric columns should be numeric for analysis. This catches values like "12.5" and "NA" that are not properly converted.**

In [ ]:
numeric_cols = [
    "body_temp",
    "pulse",
    "respiration",
    "systolic_bp",
    "diastolic_bp",
    "fio2",
    "creatinine",
    "urea",
    "uric_acid",
    "gfr",
    "cystatin",
    "wbc",
    "monocyte_count",
    "rbc",
    "hematocrit",
    "lymphocyte_count",
    "hemoglobin",
    "platelet_count",
    "basophil_count",
    "eosinophil_count",
    "neutrophil_count",
    "aptt",
    "fibrinogen",
    "hs_troponin",
    "myoglobin",
    "calcium",
    "potassium",
    "chloride",
    "sodium",
    "magnesium",
    "ck",
    "ldh",
    "bnp",
    "albumin",
    "globulin",
    "total_protein",
    "cholesterol",
    "triglyceride",
    "glucose_blood_gas",
    "lactate",
    "po2",
    "ph",
    "potassium_ion",
    "chloride_ion",
    "sodium_ion",
    "free_calcium"
]

for col in numeric_cols:
    if col in labs.columns:
        labs[col] = pd.to_numeric(labs[col], errors="coerce")

In [ ]:
display(labs.head())

In [ ]:
labs.shape

In [ ]:
labs.to_csv("cleaned_files/labs_clean.csv", index=False)
print("Saved labs_clean.csv")

# ***Hospitalization Discharge***

### **1. Read and inspect the hospitalization discharge file**

In [ ]:
hd=pd.read_csv("data_files/hospitalization_discharge.csv")

print(hd.shape)
print(hd.head())

### **2. Check for missing values and duplicates to avoid data redundancy**

In [ ]:
hd.info()
print(hd.duplicated().sum())

### **3. Remove completely empty columns as they cannot be further used for analysis**

In [ ]:
empty_columns = hd.columns[hd.isna().all()]
print(empty_columns.tolist()) 

hd = hd.drop(columns=empty_columns)


### **4. Check for column datatyes and change the columns to appropriate datatypes('admission_date' from string to datatime) for data consistency and accurate filtering for better satistical analysis**

In [ ]:
hd.dtypes
hd['admission_date'] = pd.to_datetime(hd['admission_date'], errors='coerce')



### **5. Rename columns for better readability and consistency**


In [ ]:
hd.rename(columns={
    "inpatient_number": "patient_id",
    "destinationdischarge": "discharge_destination",
    "admission_ward": "admission_ward",
    "admission_way": "admission_mode",
    "discharge_department": "discharge_department",
    "visit_times": "visit_count",
    "respiratory_support": "resp_support",
    "oxygen_inhalation": "oxygen_use",
    "dischargeday": "discharge_day",
    "admission_date": "admission_date",
    "outcome_during_hospitalization": "in_hosp_outcome",
    "death_within_28_days": "mortality_28d",
    "re_admission_within_28_days": "readmission_28d",
    "death_within_3_months": "death_3mo",
    "re_admission_within_3_months": "readmission_3mo",
    "death_within_6_months": "mortality_3mo",
    "re_admission_within_6_months": "readmission_6mo",
    "time_of_death__days_from_admission": "death_time_days",
    "readmission_time_days_from_admission": "readmission_time_days",
    "return_to_emergency_department_within_6_months": "ed_return_6mo",
    "time_to_emergency_department_within_6_months": "ed_return_time_days"
}, inplace=True)

### **6. Verify changes and load clean data**

In [ ]:
hd.info()
hd.to_csv("cleaned_files/hospitalization_discharge_clean.csv", index=False)


# ***Responsivenss***

### **1. Read and inspect responsivenes file**

In [ ]:
resp=pd.read_csv("data_files/responsivenes.csv")

print(resp.shape)
print(resp.head())

### **2. Check missing values and duplicates to avoid data redundancy**

In [ ]:
print(resp.isna().sum())
print(resp.duplicated().sum())

### **3. Remove completely empty columns if any as they cannot be used for analysis**

In [ ]:
empty_columns = resp.columns[resp.isna().all()]
print(empty_columns.tolist())

### **4. Check for column datatypes and change if not appropriate**

resp.dtypes

### **5. Rename "inpatient_number" to "patient_id" for data consistency and better readability**

In [ ]:
resp.rename(columns={
    "inpatient_number": "patient_id"}, inplace=True)

### **6.Verify changes and load clean data**

In [ ]:
resp.info()
resp.to_csv("cleaned_files/responsivenes_clean.csv", index=False)

In [ ]:
agg_prescreption = pd.read_csv("cleaned_files/prescriptions_clean.csv")

agg_prescreption = agg_prescreption.rename(columns={"inpatient_number": "patient_id"})
agg_prescreption["drug_name"] = agg_prescreption["drug_name"].astype(str).str.strip().str.lower()

patient_drug_summary = (
    agg_prescreption.groupby("patient_id")
      .agg(
          total_drugs=("drug_name", "count"),
          drugs_list=("drug_name", lambda x: ", ".join(sorted(set(x))))
      )
      .reset_index()
)

print(patient_drug_summary.head())

In [ ]:
patient_drug_summary.to_csv("cleaned_files/summarized_prescreption_clean.csv", index=False)

## **Cardiac Complications Cleanig**

### **1. Read the Csv File and inspect the data**


In [ ]:
dfcardiac = pd.read_csv("data_files/cardiac_complications.csv")

# Standardize column names for clean analysis
# Keep names lowercase and replace spaces/symbols with underscores

dfcardiac.columns = dfcardiac.columns.str.strip().str.lower()
dfcardiac.columns = dfcardiac.columns.str.replace(r"[^a-z0-9]+", "_", regex=True)

dfcardiac = dfcardiac.rename(columns={
    "inpatient_number": "patient_id",
    "type_of_heart_failure": "heart_failure_type",
    "left_ventricular_end_diastolic_diameter_lv": "lv_end_diastolic_diameter"
})

print(dfcardiac.shape)
print(dfcardiac.info())
print(dfcardiac.columns.tolist())


### **2. Renamed some column names to meaningful names.**

In [ ]:
dfcardiac = dfcardiac.rename(columns={
    "inpatient_number": "patient_id",
    "type_of_heart_failure": "heart_failure_type",
    "left_ventricular_end_diastolic_diameter_lv": "lv_end_diastolic_diameter"
})

print(dfcardiac.shape)
print(dfcardiac.info())
print(dfcardiac.columns.tolist())

### **3. Standardize all column names with lower-case and underscores.**


In [ ]:
dfcardiac.columns = dfcardiac.columns.str.strip().str.lower()
dfcardiac.columns = dfcardiac.columns.str.replace(r"[^a-z0-9]+", "_", regex=True)

### **4.Remove duplicate rows.**

In [ ]:
dfcardiac = dfcardiac.drop_duplicates()

### **5. Check missing values.**

In [ ]:
dfcardiac.isna().sum().sort_values(ascending=False)


### **6. Handle missing values based on type.**

In [ ]:
dfcardiac = dfcardiac.fillna(dfcardiac.median(numeric_only=True))

### **7. Convert object/string columns to proper format.**

In [ ]:
dfcardiac["heart_failure_type"] = (
    dfcardiac["heart_failure_type"]
    .astype(str)
    .str.strip()
)

### **8. Review and clean categorical values.**

In [ ]:
dfcardiac["heart_failure_type"] = dfcardiac["heart_failure_type"].str.lower().str.strip()


### **9. Check for impossible or invalid numeric values.**

In [ ]:
for col in dfcardiac.select_dtypes(include=["number"]).columns:
    print(col, dfcardiac[col].describe())

### **10. Final dataset validation.**

In [ ]:
dfcardiac.shape
dfcardiac.isna().sum().sum()
dfcardiac.dtypes

In [ ]:

dfcardiac.to_csv("cleaned_files/cardiac_cleaned.csv", index=False)





### **Merged Datasets**

In [ ]:
from fileinput import filename

data_folder = "cleaned_files"
#data_folder = Path(r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\PyCoders_Github\Team3_PyCoders_PythonHackathon_SEP2026\cleaned_files")

files = {
    "demography": "demography_clean.csv",
    "history": "patienthistory_clean.csv",
    "hospital":"hospitalization_discharge_clean.csv",
    "labs": "labs_clean.csv",
    "cardiac_complications": "cardiac_cleaned.csv",
    "responsiveness": "responsivenes_clean.csv",
    "summarized_prescription": "summarized_prescreption_clean.csv"
}

datasets = {name: pd.read_csv(f"{data_folder}/{filename}") for name, filename in files.items()}

# Check the actual column names before choosing the merge key
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

In [ ]:
from fileinput import filename


merged = datasets["demography"].copy()
merged.head()




In [ ]:
for name in ["history", "hospital", "summarized_prescription", "cardiac_complications", "responsiveness", "labs"]:
    merged = merged.merge(
        datasets[name],
        on="patient_id",
        how="left",
        validate="one_to_one",
        suffixes=("", f"_{name}")
    )
    print(f"After {name}: {merged.shape}")

merged.to_csv(f"{data_folder}/heart_failure_merged.csv", index=False)
display(merged.head())

In [ ]:
merged.shape